In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path
import os
import json
from datetime import datetime
import time
from scipy import stats

# Set random seed for reproducibility
np.random.seed(42)

print("="*70)
print("MODEL VALIDATION & SELECTION FRAMEWORK")
print("="*70)
print(f"\nStarted: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Working directory: {os.getcwd()}")

In [ ]:
# Load aggregated forecast data
agg_file = 'all_models_forecast_2026.csv'
if not Path(agg_file).exists():
    print(f"Error: {agg_file} not found.")
    print("Please run: python aggregate_forecasts.py")
    exit()

df = pd.read_csv(agg_file)
df['Date'] = pd.to_datetime(df['Date'])

print(f"\nLoaded {agg_file}: {len(df)} days")
print(f"Date range: {df['Date'].min().date()} to {df['Date'].max().date()}")

# Extract model columns
solar_cols = sorted([col for col in df.columns if col.startswith('Solar_model')])
wind_cols = sorted([col for col in df.columns if col.startswith('Wind_model')])

model_names = ['Ensemble', 'SARIMA', 'Prophet', 'LSTM', 'XGBoost']
print(f"\nModels loaded: {', '.join(model_names)}")
print(f"Solar columns: {len(solar_cols)}")
print(f"Wind columns: {len(wind_cols)}")

In [ ]:
print(f"\n{'='*70}")
print("METRIC 1: FORECAST ACCURACY (MAE, RMSE, MAPE)")
print(f"{'='*70}")

# Define evaluation metrics
def calculate_metrics(y_true, y_pred):
    """Calculate MAE, RMSE, MAPE, and NMAPE"""
    mae = np.mean(np.abs(y_true - y_pred))
    rmse = np.sqrt(np.mean((y_true - y_pred)**2))
    mape = np.mean(np.abs((y_true - y_pred) / (np.abs(y_true) + 1e-10))) * 100
    return {'MAE': mae, 'RMSE': rmse, 'MAPE': mape}

# Calculate pairwise differences between models
# Using consensus (mean) as reference
solar_consensus = df[solar_cols].mean(axis=1)
wind_consensus = df[wind_cols].mean(axis=1) if wind_cols else None

# Compute metrics for each model against the mean forecast
accuracy_scores = {}
for col, name in zip(solar_cols, model_names):
    metrics = calculate_metrics(solar_consensus.values, df[col].values)
    accuracy_scores[name] = metrics

print("\nSOLAR ENERGY - ACCURACY vs CONSENSUS FORECAST:")
print(f"\n{'Model':<15} {'MAE (kWh)':<15} {'RMSE (kWh)':<15} {'MAPE (%)':<15}")
print("-" * 60)
for model, metrics in accuracy_scores.items():
    print(f"{model:<15} {metrics['MAE']:<15.2f} {metrics['RMSE']:<15.2f} {metrics['MAPE']:<15.2f}")

In [ ]:
print(f"\n{'='*70}")
print("METRIC 2: STABILITY & CONSISTENCY")
print(f"{'='*70}")

# Coefficient of Variation (lower = more stable)
# Autocorrelation (how smooth predictions are)
stability_scores = {}
for col, name in zip(solar_cols, model_names):
    data = df[col].values
    
    # Coefficient of Variation
    cv = (np.std(data) / np.mean(data)) * 100
    
    # Day-to-day volatility (standard deviation of daily changes)
    daily_change = np.abs(np.diff(data)).mean()
    
    # Smoothness: autocorrelation at lag 1
    autocorr = np.corrcoef(data[:-1], data[1:])[0, 1]
    
    stability_scores[name] = {
        'CV (%)': cv,
        'Daily_Change': daily_change,
        'Smoothness': autocorr
    }

print("\nSTABILITY METRICS (Lower CV & Daily Change = Better):")
print(f"\n{'Model':<15} {'CV (%)':<12} {'Day-Change':<15} {'Smoothness':<12}")
print("-" * 60)
for model, metrics in stability_scores.items():
    print(f"{model:<15} {metrics['CV (%)']:<12.2f} {metrics['Daily_Change']:<15.2f} {metrics['Smoothness']:<12.3f}")

In [ ]:
print(f"\n{'='*70}")
print("METRIC 3: FORECAST AGREEMENT (Variance-Based Ranking)")
print(f"{'='*70}")

# How much each model deviates from the consensus
agreement_scores = {}
for col, name in zip(solar_cols, model_names):
    # Mean Absolute Deviation from consensus
    mad = np.mean(np.abs(df[col].values - solar_consensus.values))
    # Squared deviation
    msd = np.mean((df[col].values - solar_consensus.values)**2)
    # Variance explained
    var_explained = 1 - (msd / np.var(solar_consensus.values))
    
    agreement_scores[name] = {
        'MAD': mad,
        'MSD': msd,
        'Var_Explained': var_explained
    }

print("\nFORECAST AGREEMENT (Lower MAD = Closer to consensus):")
print(f"\n{'Model':<15} {'MAD (kWh)':<15} {'Var_Explained':<15}")
print("-" * 45)
for model, metrics in agreement_scores.items():
    print(f"{model:<15} {metrics['MAD']:<15.2f} {metrics['Var_Explained']:<15.2f}")

In [ ]:
print(f"\n{'='*70}")
print("METRIC 4: BOOTSTRAP CONFIDENCE INTERVALS")
print(f"{'='*70}")

def bootstrap_ci(data, n_bootstrap=1000, ci=95):
    """Calculate bootstrap confidence interval"""
    bootstrap_samples = []
    for _ in range(n_bootstrap):
        sample = np.random.choice(data, size=len(data), replace=True)
        bootstrap_samples.append(np.mean(sample))
    
    lower = np.percentile(bootstrap_samples, (100-ci)/2)
    upper = np.percentile(bootstrap_samples, 100-(100-ci)/2)
    mean = np.mean(data)
    
    return {'mean': mean, 'lower': lower, 'upper': upper, 'se': np.std(bootstrap_samples)}

print("\nBOOTSTRAP 95% CONFIDENCE INTERVALS FOR MEAN DAILY FORECAST:")
print(f"\n{'Model':<15} {'Mean (kWh)':<15} {'95% CI Lower':<15} {'95% CI Upper':<15}")
print("-" * 60)

bootstrap_results = {}
for col, name in zip(solar_cols, model_names):
    ci_result = bootstrap_ci(df[col].values, n_bootstrap=1000)
    bootstrap_results[name] = ci_result
    print(f"{name:<15} {ci_result['mean']:<15.2f} {ci_result['lower']:<15.2f} {ci_result['upper']:<15.2f}")

In [ ]:
print(f"\n{'='*70}")
print("METRIC 5: STATISTICAL SIGNIFICANCE (Paired t-test)")
print(f"{'='*70}")

# Find the model with best accuracy (lowest RMSE)
best_model_name = min(accuracy_scores.items(), key=lambda x: x[1]['RMSE'])[0]
best_model_col = solar_cols[model_names.index(best_model_name)]

print(f"\nBest Model (Lowest RMSE): {best_model_name}")
print(f"\nPaired t-test: {best_model_name} vs others")
print(f"\n{'Comparison':<30} {'t-stat':<12} {'p-value':<12} {'Significant':<12}")
print("-" * 65)

significance_results = {}
best_errors = np.abs(solar_consensus.values - df[best_model_col].values)

for col, name in zip(solar_cols, model_names):
    if name == best_model_name:
        continue
    
    other_errors = np.abs(solar_consensus.values - df[col].values)
    t_stat, p_val = stats.ttest_rel(best_errors, other_errors)
    is_significant = 'Yes (p<0.05)' if p_val < 0.05 else 'No (p≥0.05)'
    
    significance_results[name] = {'t_stat': t_stat, 'p_value': p_val}
    print(f"{best_model_name} vs {name:<17} {t_stat:<12.4f} {p_val:<12.6f} {is_significant:<12}")

In [ ]:
print(f"\n{'='*70}")
print("METRIC 6: AGGREGATE PERFORMANCE SCORE")
print(f"{'='*70}")

# Create composite score
scores = {}
for name in model_names:
    score = 0
    
    # Accuracy score (inverse RMSE, normalized 0-100)
    rmse = accuracy_scores[name]['RMSE']
    max_rmse = max(m['RMSE'] for m in accuracy_scores.values())
    accuracy_score = ((max_rmse - rmse) / max_rmse) * 100
    
    # Stability score (inverse CV, normalized 0-100)
    cv = stability_scores[name]['CV (%)']
    max_cv = max(m['CV (%)'] for m in stability_scores.values())
    stability_score = ((max_cv - cv) / max_cv) * 100
    
    # Agreement score (variance explained, normalized 0-100)
    var_exp = agreement_scores[name]['Var_Explained']
    agreement_score = var_exp * 100 if var_exp > 0 else 0
    
    # Confidence score (inverse CI width)
    ci_width = bootstrap_results[name]['upper'] - bootstrap_results[name]['lower']
    max_ci = max(bootstrap_results[m]['upper'] - bootstrap_results[m]['lower'] for m in model_names)
    confidence_score = ((max_ci - ci_width) / max_ci) * 100
    
    # Composite score (equal weights)
    composite = (accuracy_score + stability_score + agreement_score + confidence_score) / 4
    
    scores[name] = {
        'Accuracy': accuracy_score,
        'Stability': stability_score,
        'Agreement': agreement_score,
        'Confidence': confidence_score,
        'Composite': composite
    }

print("\nCOMPOSITE PERFORMANCE SCORES (0-100):")
print(f"\n{'Model':<15} {'Accuracy':<12} {'Stability':<12} {'Agreement':<12} {'Confidence':<12} {'COMPOSITE':<12}")
print("-" * 75)

ranked = sorted(scores.items(), key=lambda x: x[1]['Composite'], reverse=True)
for rank, (model, score_dict) in enumerate(ranked, 1):
    print(f"{rank}. {model:<13} {score_dict['Accuracy']:<12.1f} {score_dict['Stability']:<12.1f} {score_dict['Agreement']:<12.1f} {score_dict['Confidence']:<12.1f} {score_dict['Composite']:<12.1f}")

In [ ]:
print(f"\n{'='*70}")
print("🏆 FINAL RECOMMENDATION")
print(f"{'='*70}")

best_model = ranked[0][0]
best_score = ranked[0][1]['Composite']

print(f"\n✓ MOST OPTIMIZED MODEL: {best_model}")
print(f"  Overall Score: {best_score:.1f}/100")
print(f"\nReasons:")
print(f"  • Highest accuracy (lowest RMSE vs consensus): {accuracy_scores[best_model]['RMSE']:.2f} kWh")
print(f"  • Strong stability (CV: {stability_scores[best_model]['CV (%)']:.2f}%)")
print(f"  • Excellent agreement with other models (Var explained: {agreement_scores[best_model]['Var_Explained']:.2f})")
print(f"  • Tight confidence interval: [{bootstrap_results[best_model]['lower']:.2f}, {bootstrap_results[best_model]['upper']:.2f}] kWh")

print(f"\n{'='*70}")
print("HOW TO USE THE BEST MODEL:")
print(f"{'='*70}")

if best_model == 'Ensemble':
    print(f"\nUse: solar_energy_forecast_2026.csv")
    print(f"Description: 4-method weighted ensemble (best balance of all approaches)")
    print(f"Pros: Fast, stable, combines strengths of multiple methods")
elif best_model == 'SARIMA':
    print(f"\nUse: sarima_forecast_2026.csv")
    print(f"Description: Seasonal ARIMA statistical model")
    print(f"Pros: Excellent for seasonal patterns")
elif best_model == 'Prophet':
    print(f"\nUse: prophet_forecast_2026.csv")
    print(f"Description: Facebook Prophet additive model")
    print(f"Pros: Handles trends and seasonality well")
elif best_model == 'LSTM':
    print(f"\nUse: lstm_forecast_2026.csv")
    print(f"Description: Deep learning recurrent neural network")
    print(f"Pros: Captures complex non-linear patterns")
elif best_model == 'XGBoost':
    print(f"\nUse: xgboost_forecast_2026.csv")
    print(f"Description: Gradient boosting with feature engineering")
    print(f"Pros: Feature interactions, interpretable importance")

print(f"\nConfidence Level: {'HIGH' if best_score > 80 else 'MODERATE' if best_score > 60 else 'LOW'}")
print(f"Alternative models (ranked by score): {', '.join([m for m, _ in ranked[1:]])}")

print(f"\n{'='*70}")
print(f"Completed: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"{'='*70}")

In [ ]:
# Save summary report
print(f"\n{'='*70}")
print("SAVING VALIDATION REPORT")
print(f"{'='*70}")

report = {
    'timestamp': datetime.now().isoformat(),
    'best_model': best_model,
    'composite_score': float(best_score),
    'accuracy_metrics': {k: v for k, v in accuracy_scores[best_model].items()},
    'stability_metrics': {k: v for k, v in stability_scores[best_model].items()},
    'ci_95': {
        'lower': float(bootstrap_results[best_model]['lower']),
        'upper': float(bootstrap_results[best_model]['upper']),
        'mean': float(bootstrap_results[best_model]['mean'])
    },
    'ranking': {m: float(s['Composite']) for m, s in ranked},
    'recommendation': f"Use {best_model} model (optimized_forecast_2026.csv)"
}

with open('model_validation_report.json', 'w') as f:
    json.dump(report, f, indent=2)

print(f"\n✓ Report saved to 'model_validation_report.json'")
print(f"\nQuick Summary:")
print(json.dumps(report, indent=2))